# Fine-tune XLM-RoBERTa-large-squad2 on ViquAD

Transfer learning from SQuAD 2.0 to Vietnamese ViquAD.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas pyarrow sentencepiece
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os, torch, pandas as pd, numpy as np
from datasets import Dataset
from transformers import (
    XLMRobertaTokenizerFast,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    logging as hf_logging
)
hf_logging.set_verbosity_error()

In [ ]:
def find_char_span(context: str, answer: str):
    if not answer or not context:
        return -1, -1
    c_chars = [(char, i) for i, char in enumerate(context) if char not in (" ", "_")]
    a_chars = [char for char in answer if char not in (" ", "_")]
    c_str = "".join([x[0] for x in c_chars])
    a_str = "".join(a_chars)
    idx = c_str.find(a_str)
    if idx == -1:
        return -1, -1
    start_char_idx = c_chars[idx][1]
    end_char_idx = c_chars[idx + len(a_str) - 1][1] + 1
    return start_char_idx, end_char_idx

def load_qa_dataset(file_path: str) -> Dataset:
    df = pd.read_parquet(file_path)
    if "context_segmented" in df.columns:
        df = df.drop(columns=["context", "question", "answer_text"], errors="ignore")
        df = df.rename(columns={
            "context_segmented": "context",
            "question_segmented": "question",
            "answer_text_segmented": "answer_text"
        })
    data_dict = {
        "id": df["id"].astype(str).tolist(),
        "context": df["context"].astype(str).tolist(),
        "question": df["question"].astype(str).tolist(),
        "answer_text": df["answer_text"].fillna("").astype(str).tolist(),
        "answer_start": df["answer_start"].fillna(-1).astype(int).tolist(),
    }
    return Dataset.from_dict(data_dict)

def prepare_train_features(examples, tokenizer, max_seq_len=384, doc_stride=128):
    examples["question"] = [q[:150] for q in examples["question"]]
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_seq_len,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        raw_context = examples["context"][sample_index]
        raw_answer = examples["answer_text"][sample_index]
        raw_start = examples["answer_start"][sample_index]
        if raw_start == -1 or not raw_answer.strip():
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue
        start_char, end_char = find_char_span(raw_context, raw_answer)
        if start_char == -1 or end_char == -1:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue
        token_start_index = 0
        while token_start_index < len(sequence_ids) and sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while token_end_index >= 0 and sequence_ids[token_end_index] != 1:
            token_end_index -= 1
        if token_start_index > token_end_index or not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            tokenized_examples["start_positions"].append(token_start_index - 1)
            while token_end_index >= 0 and offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            tokenized_examples["end_positions"].append(token_end_index + 1)
    return tokenized_examples

def rebalance_features(ds: Dataset, cls_ratio: float = 2.0) -> Dataset:
    cls_id = 0
    has_span = np.array([s != cls_id or e != cls_id for s, e in zip(ds["start_positions"], ds["end_positions"])])
    span_idx = np.where(has_span)[0]
    cls_idx = np.where(~has_span)[0]
    target_cls = min(len(cls_idx), int(len(span_idx) * cls_ratio))
    keep_cls = np.random.choice(cls_idx, target_cls, replace=False)
    keep = np.concatenate([span_idx, keep_cls])
    np.random.shuffle(keep)
    return ds.select(keep)

def compute_metrics(eval_pred):
    logits_s, logits_e = eval_pred.predictions
    start_pos, end_pos = eval_pred.label_ids
    cls_id = 0
    mask = (start_pos != cls_id) | (end_pos != cls_id)
    if mask.sum() == 0:
        return {"answerable_em": 0.0}
    pred_s = logits_s[mask].argmax(-1)
    pred_e = logits_e[mask].argmax(-1)
    em = (pred_s == start_pos[mask]) & (pred_e == end_pos[mask])
    return {"answerable_em": float(em.mean())}

In [ ]:
train_path = "viquad_train_segmented.parquet"
val_path = "viquad_val_segmented.parquet"
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if "train" in f and f.endswith(".parquet"):
                train_path = os.path.join(root, f)
            elif "val" in f and f.endswith(".parquet"):
                val_path = os.path.join(root, f)
print(f"Train: {train_path}")
print(f"Val:   {val_path}")
train_dataset = load_qa_dataset(train_path)
val_dataset = load_qa_dataset(val_path)
print(f"Train: {len(train_dataset)} samples, Val: {len(val_dataset)} samples")

In [ ]:
model_name = "deepset/xlm-roberta-large-squad2"
print(f"Loading tokenizer from {model_name}...")
tokenizer = XLMRobertaTokenizerFast.from_pretrained(model_name)
max_seq_len = 384
doc_stride = 128
print("Tokenizing...")
tokenized_train = train_dataset.map(
    lambda x: prepare_train_features(x, tokenizer, max_seq_len, doc_stride),
    batched=True, remove_columns=train_dataset.column_names
)
tokenized_train = rebalance_features(tokenized_train, cls_ratio=1.0)
tokenized_val = val_dataset.map(
    lambda x: prepare_train_features(x, tokenizer, max_seq_len, doc_stride),
    batched=True, remove_columns=val_dataset.column_names
)
print(f"Features - Train: {len(tokenized_train)}, Val: {len(tokenized_val)}")

In [ ]:
print("Loading XLM-RoBERTa-large-squad2...")
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
output_dir = "./results_xlmr"
eval_key = "eval_strategy" if hasattr(TrainingArguments("./tmp"), "eval_strategy") else "evaluation_strategy"
kwargs_args = {
    "output_dir": output_dir,
    eval_key: "steps",
    "eval_steps": 500,
    "learning_rate": 1e-5,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 16,
    "num_train_epochs": 3,
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "cosine",
    "weight_decay": 0.01,
    "save_total_limit": 1,
    "logging_steps": 50,
    "save_strategy": "steps",
    "save_steps": 500,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_answerable_em",
    "greater_is_better": True,
    "report_to": "none",
"fp16": torch.cuda.is_available(),
    "gradient_checkpointing": True,
    "optim": "adafactor",
}
training_args = TrainingArguments(**kwargs_args)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
final_model_dir = "./xlm-roberta-large-viquad"
print(f"Saving best model to {final_model_dir}...")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
!zip -r xlm-roberta-large-viquad.zip {final_model_dir}
print("Done! Download xlm-roberta-large-viquad.zip")

In [ ]:
# Quick verify (add project root to path if running in Colab)
import sys
sys.path.insert(0, "/content/")
from reader.evaluate import evaluate
evaluate(
    model_path="./xlm-roberta-large-viquad",
    data_variant="segmented",
    subset_size=100,
    use_cpu=True,
    output_file="./xlmr_eval.json"
)
# Target: Answerable EM >= 70%